# 05 - TabTransformer | Sensor 1

Trains TabTransformer from scratch on Sensor 1. This model is then fine-tuned on Sensors 2 and 3.

Split is 70/15/15: validation picks the checkpoint, test is used once.

In [1]:
import sys
sys.path.append("..")

import pandas as pd

from src.common import (
    PROCESSED_DIR, MODELS_DIR, TARGET,
    CATEGORICAL_COLS, CONTINUOUS_COLS,
    three_way_split, report_and_save,
)

from pytorch_tabular import TabularModel
from pytorch_tabular.config import DataConfig, OptimizerConfig, TrainerConfig
from pytorch_tabular.models import TabTransformerConfig

In [2]:
df = pd.read_csv(f"{PROCESSED_DIR}/sensor1_processed.csv")
train_df, val_df, test_df = three_way_split(df)
print("train", train_df.shape, "| val", val_df.shape, "| test", test_df.shape)

train (3426610, 15) | val (734274, 15) | test (734274, 15)


In [3]:
data_config = DataConfig(
    target=[TARGET],
    continuous_cols=CONTINUOUS_COLS,
    categorical_cols=CATEGORICAL_COLS,
)
trainer_config = TrainerConfig(
    max_epochs=20,
    batch_size=1024,
    accelerator="auto",
    early_stopping="valid_loss",
    early_stopping_patience=3,
    checkpoints="valid_loss",
    load_best=True,
    seed=42,
)
model_config = TabTransformerConfig(task="classification", learning_rate=1e-3)
model = TabularModel(
    data_config=data_config,
    model_config=model_config,
    optimizer_config=OptimizerConfig(),
    trainer_config=trainer_config,
)

In [4]:
model.fit(train=train_df, validation=val_df)   # validation = val, NOT test
model.save_model(f"{MODELS_DIR}/tabtransformer_pretrained_sensor1")

In [5]:
pred = model.predict(test_df)
report_and_save("TabTransformer", "sensor1", test_df[TARGET], pred[f"{TARGET}_prediction"])

TabTransformer | sensor1
accuracy         : 0.9962
precision_botnet : 0.9436
recall_botnet    : 0.9291
f1_botnet        : 0.9363
f1_macro         : 0.9672
f1_weighted      : 0.9962

              precision    recall  f1-score   support

           0     0.9978    0.9983    0.9981    712374
           1     0.9436    0.9291    0.9363     21900

    accuracy                         0.9962    734274
   macro avg     0.9707    0.9637    0.9672    734274
weighted avg     0.9962    0.9962    0.9962    734274

Confusion matrix: [[711157, 1217], [1552, 20348]]


{'accuracy': 0.9962289281657801,
 'precision_botnet': 0.9435659633665662,
 'recall_botnet': 0.9291324200913242,
 'f1_botnet': 0.9362935695387093,
 'f1_macro': 0.9671752624089027,
 'f1_weighted': 0.9962148389196519,
 'confusion_matrix': [[711157, 1217], [1552, 20348]]}